## Use superdense coding protocol to exchange 
### Definition  
A protocol that enables the transmis,sion of 2 classical bits by transmitting 1 qubit, (only if the Tx and Rx share a pair of qubitd in a bell state)   
### The idea:
1- We create a Bell pair (2 maximally entangled qubits), (q0 for client, and q1 for server)  
2- Client encodes 2 classical bits that he wants to transmit by applying the gates (x and z)  on 1 qubit  
3- Client sends their qubit (that has to be serialized to become a state) to the server with sockets (local for now)  
4- Server decodes the qubit by applying Bell measurement to read the 2 classical bits

### Potential link between superdense coding and QKD
One of the protocols proposed in QKD, _E91_  but also a variant of the QKD, the Quantum Secure Direct Communication that kind of uses the same mecanism as the superdense coding (the _two step QKDC protocol_)

In [1]:
from qiskit import QuantumCircuit, ClassicalRegister
from qiskit.quantum_info import Statevector
import socket
import threading
import time
import pickle

## More on Bell state: 
 It is a maximally entangled state of 2 qubits (A and B). 
 _Maximally entangled_ means that they are fully correlated (if i know everything about A, i know everything about B).  
 We have 4 bell states: Phi plus, Phi minus, Psi plus and Psi minus  

## Statevector 
used to simulate the quantum channel here, and generally used to simulate the quantum states measurements 


In [2]:
def prepare_bell_state():
    #create a bell pair of qubits
    qc=QuantumCircuit(2)
    #apply hadamard gate on q0 for superpostiion
    qc.h(0)
    #cnot gate to entangle q0 and q1
    qc.cx(0,1)
    #visualisation
    print(qc.draw(output='text'))
    return Statevector.from_instruction(qc)

In [3]:
bell_state=prepare_bell_state()

     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘


In [4]:
bell_state

Statevector([0.70710678+0.j, 0.        +0.j, 0.        +0.j,
             0.70710678+0.j],
            dims=(2, 2))


Client applies the gates to q0 based on the 2-bit message: 
if msg(b1b0)== 00       => qc.i()  
if msg(b1b0)== 01       => qc.x()  
if msg(b1b0)== 10       => qc.z()  
if msg(b1b0)== 11       => qc.x(), qc.z()  

In [5]:
def client(bell_vector, b1, b0):
    #encode the msg to be sent
    qc=QuantumCircuit(2)
    if b0==1:
        #x gate to flip the bit (gate applied to q0)
        qc.x(0)
    if b1==1:
        #z gate to flip the phase (gate applied to q0)
        qc.z(0)
    
    return bell_vector.evolve(qc), qc


In [6]:
bit1=1
bit0=1


In [7]:
def server(sv):
    #server will decode the statevector recvd
    #apply reverse bell measures (cnot, h, mesure)
    #returns b1b0 as the decoded classical bits

    qc=QuantumCircuit(2,2)
    #cnot gate to detangle
    qc.cx(0,1)
    qc.h(0)
  

    #simulation measurement
    evolved=sv.evolve(qc)
   
    counts=evolved.sample_counts(shots=1)
    bitstring=list(counts.keys())[0] 

    b0 = int(bitstring[-2]) 
    b1 = int(bitstring[-1]) 
    decode_qc = qc.copy()
    decode_qc.add_register(ClassicalRegister(2))
    decode_qc.measure([0,1],[0,1])

    return b1, b0, decode_qc


In [8]:
server_result = {} 
def run_server():
    #server: listen for 1 qubit (sv), decodes it and give result"""
    host=socket.gethostname()
    port=5000
    s=socket.socket()
    s.bind((host, port))
    s.listen(1)
    print(f"Server Listening on{port}...")

    conn, addr=s.accept()
    with conn:
        print(f"Server Connected by {addr}")
        data=b""
        while True:
            d=conn.recv(4096)
            if not d:
                break
            data +=d

        #deserializer the received Statevector
        received_sv=pickle.loads(data)
        print(f"Server Received Statevector: {received_sv}")

        #decode
        b1, b0, decode_qc=server(received_sv)
        server_result['b1']=b1
        server_result['b0']=b0
        server_result['circuit']=decode_qc
        print(f"Server Decoded bits: b1={b1}, b0={b0}")
        print("Server Decoding circuit:")
        print(decode_qc.draw('text'))

In [18]:
start=time.perf_counter()
encoded_state, encoding_circuit=client(bell_state, bit1, bit0)
mid1=time.perf_counter()

In [10]:
print(f"Client wants to send bits: b1={bit1}, b0={bit0}")
print("\nEncoding circuit applied to q0:")
print(encoding_circuit.draw('text'))
print(f"\nEncoded Statevector: {encoded_state}")

Client wants to send bits: b1=1, b0=1

Encoding circuit applied to q0:
     ┌───┐┌───┐
q_0: ┤ X ├┤ Z ├
     └───┘└───┘
q_1: ──────────
               

Encoded Statevector: Statevector([ 0.        +0.j, -0.70710678+0.j,  0.70710678+0.j,
             -0.        +0.j],
            dims=(2, 2))


In [30]:
#start server in bg thread
server_thread=threading.Thread(target=run_server,daemon=True)
server_thread.start()
time.sleep(0.5)  #give server time to start
print("Server thread started")

Server Listening on5000...
Server thread started


In [28]:
def client_send(statevector):
    host=socket.gethostname()
    port=5000
    """Client: serialize and send Statevector to Server via socket."""
    payload=pickle.dumps(statevector)
    c=socket.socket()
    c.connect((host, port))
    c.sendall(payload)
    print(f"Client Sent encoded qubit to server")
    print(f"{len(payload)} bytes were sent")


In [31]:
client_send(encoded_state)
end=time.perf_counter()

Client Sent encoded qubit to serverServer Connected by ('192.168.41.1', 55917)

433 bytes were sent
Server Received Statevector: Statevector([0.70710678+0.j, 0.        +0.j, 0.        +0.j,
             0.70710678+0.j],
            dims=(2, 2))
Server Decoded bits: b1=0, b0=0
Server Decoding circuit:
             ┌───┐┌─┐
   q_0: ──■──┤ H ├┤M├
        ┌─┴─┐└┬─┬┘└╥┘
   q_1: ┤ X ├─┤M├──╫─
        └───┘ └╥┘  ║ 
   c: 2/═══════╩═══╩═
               1   0 
c202: 2/═════════════
                     


In [32]:
#wait for server to finish
server_thread.join(timeout=5)
print("Server thread finished")

Server thread finished


In [23]:
print("=======================================")
print("      SUPERDENSE CODING RESULTS")
print("=======================================")
print(f"  Message sent from client: b1={bit1}, b0={bit0}")
print(f"  Message received:         b1={server_result.get('b1','?')}, b0={server_result.get('b0','?')}")
print("=======================================")

if server_result.get('b1')==bit1 and server_result.get('b0')==bit0:
    print("SUCCESS: Bits transmitted correctly!")
else:
    print("x not working")

      SUPERDENSE CODING RESULTS
  Message sent from client: b1=1, b0=1
  Message received:         b1=1, b0=1
SUCCESS: Bits transmitted correctly!


In [33]:
print("Encoding time:", mid1 - start)
print("Decoding time:", end - mid1)
print("Total time:", end - start)

Encoding time: 0.00015180000082182232
Decoding time: 37.62758799999938
Total time: 37.6277398000002


In [27]:
#to evaluate accuracy
import random

N = 100
success = 0

for _ in range(N):
    bit1 = random.randint(0,1)
    bit0 = random.randint(0,1)
    start = time.perf_counter()
    encoded_state, _ = client(bell_state, bit1, bit0)
    mid1 = time.perf_counter()
    received_b1, received_b0, _ = server(encoded_state)
    
    end = time.perf_counter()


    if received_b1 == bit1 and received_b0 == bit0:
        success += 1
print("Encoding time:", mid1 - start)
print("Decoding time:", end - mid1)
print("Total time:", end - start)
print(f"Success rate: {success/N * 100}%")

Encoding time: 0.00015180000082182232
Decoding time: 0.0008797000000413391
Total time: 0.0010315000008631614
Success rate: 100.0%
